In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import ast
import string

In [2]:
# Load the training data
train_df = pd.read_csv("../data/train_translated_tokenized.csv")
train_df.rename(columns={train_df.columns[0]: "real_dataset_index"}, inplace=True)

In [3]:
# Display n gram in table format
def generate_n_gram_table(
    df: pd.DataFrame,
    n: int = 2,
    normalize: bool = True,
) -> pd.DataFrame:
    n_gram_counter = Counter()

    for tokens in df:
        tokens = ["<START>"] * (n - 1) + tokens + ["<END>"] * (n - 1)
        n_gram_counter.update(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))
    
    df_matrix = pd.DataFrame(
        [(ngram[:-1], ngram[-1], count) for ngram, count in n_gram_counter.items()],
        columns=["prefix", "next_word", "count"]
    )
    df_matrix["prefix"] = df_matrix["prefix"].apply(lambda x: " ".join(x))
    
    pivot_table = df_matrix.pivot_table(
        index="prefix",
        columns="next_word",
        values="count",
        fill_value=0
    )
    
    pivot_table["<UNK>"] = 0
    if n >= 2:
        unk_row = pd.Series(0, index=pivot_table.columns, name="<UNK>")
        pivot_table = pd.concat([pivot_table, unk_row.to_frame().T])


    pivot_table = pivot_table + 1
    
    if normalize:
        pivot_table = pivot_table.div(pivot_table.sum(axis=1), axis=0)
    
    return pivot_table

### Korean

In [ ]:
lang="ko"
lang_df = train_df[train_df["lang"] == lang].copy()


remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)

lang_df["question_stripped"] = lang_df["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

# UNIGRAM
korean_unigram = generate_n_gram_table(lang_df["question_stripped"], n=1)
#korean_bigram.to_parquet("../data/bi_grams/korean_unigram.parquet")

# BIGRAM
korean_bigram = generate_n_gram_table(lang_df["question_stripped"], n=2)
#korean_bigram.to_parquet("../data/bi_grams/korean_bigram.parquet")


### Telugu

In [5]:
lang="te"
lang_df = train_df[train_df["lang"] == lang].copy()

remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)

lang_df["question_stripped"] = lang_df["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

# UNIGRAM
telugu_unigram = generate_n_gram_table(lang_df["question_stripped"], n=1)
#telugu_bigram.to_parquet("../data/bi_grams/telugu_unigram.parquet")

# BIGRAM
telugu_bigram = generate_n_gram_table(lang_df["question_stripped"], n=2)
#telugu_bigram.to_parquet("../data/bi_grams/telugu_bigram.parquet")

### Arabic

In [6]:
lang="ar"
lang_df = train_df[train_df["lang"] == lang].copy()

remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)

lang_df["question_stripped"] = lang_df["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

# UNIGRAM
arabic_unigram = generate_n_gram_table(lang_df["question_stripped"], n=1)
#arabic_unigram.to_parquet("../data/bi_grams/arabic_unigram.parquet")

# BIGRAM
arabic_bigram = generate_n_gram_table(lang_df["question_stripped"], n=2)
#arabic_bigram.to_parquet("../data/bi_grams/arabic_bigram.parquet")

### Context

In [7]:
remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)

context_series = train_df["context_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)

# UNIGRAM
context_unigram = generate_n_gram_table(context_series, n=1)
#context_unigram.to_parquet("../data/bi_grams/context_unigram.parquet")

# BIGRAM
context_bigram = generate_n_gram_table(context_series, n=2)
#context_bigram.to_parquet("../data/bi_grams/context_bigram.parquet")

/var/folders/wp/3lqqvgpd7mz76kqxnc2932700000gn/T/ipykernel_82284/1907471451.py:19: PerformanceWarning: The following operation may generate 2216338084 cells in the resulting pandas object.
  pivot_table = df_matrix.pivot_table(


In [8]:
# Load validation data
remove_chars = string.punctuation + "؟\n"
translator = str.maketrans('', '', remove_chars)

val_df = pd.read_csv("../data/validation_translated.csv")
val_df.rename(columns={val_df.columns[0]: "real_dataset_index"}, inplace=True)

In [10]:
# Determine probabilies for each token in a sequence
def n_probs(bigram: pd.DataFrame, sequence: list) -> float:
    if "<START>" not in bigram.index:
        previous = ""
        unigram = True
    else:
        previous = "<START>"
        unigram = False
    
    probs = [] 
    
    if not unigram:
        sequence.append("<END>")
        for word in sequence:
            try:
                probs.append(bigram.loc[previous, word])
            except KeyError as _:
                probs.append(bigram.loc["<UNK>", "<UNK>"])
            previous = word
    else:
        for word in sequence:
            try:
                probs.append(bigram[word])
            except KeyError as _:
                probs.append(bigram["<UNK>"])
    return np.array(probs)

### Compute Perplexities

In [ ]:
## Probabilies for validation set
# Korean
val_df_k = val_df[val_df["lang"] == "ko"]
korean_val = val_df_k["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)
probs_uni_korean = []
probs_bi_korean = []
for row in korean_val:
    probs_uni_korean.append(n_probs(korean_unigram, row))
    probs_bi_korean.append(n_probs(korean_bigram, row))


# Telugu
val_df_t = val_df[val_df["lang"] == "te"]
telugu_val = val_df_t["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)
probs_uni_telugu = []
probs_bi_telugu = []
for row in telugu_val:
    probs_uni_telugu.append(n_probs(telugu_unigram, row))
    probs_bi_telugu.append(n_probs(telugu_bigram, row))

# Arabic
val_df_a = val_df[val_df["lang"] == "ar"]
arabic_val = val_df_a["question_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)
probs_uni_arabic = []
probs_bi_arabic = []
for row in arabic_val:
    probs_uni_arabic.append(n_probs(arabic_unigram, row))
    probs_bi_arabic.append(n_probs(arabic_bigram, row))

# Context
context_val = val_df["context_stripped"].apply(
    lambda x: [t.translate(translator).strip().lower() for t in ast.literal_eval(x) if t.translate(translator).strip()]
)
probs_uni_context = []
probs_bi_context = []
for row in context_val:
    probs_uni_context.append(n_probs(context_unigram, row))
    probs_bi_context.append(n_probs(context_bigram, row))


In [43]:
# Perplexity
def perplexity(probs: list) -> float:
    probs = np.array(probs)
    N = len(probs)
    perplexity = (1/probs.prod())**(1/N)
    return float(perplexity)

# Korean
perplex_k = []
counter_k = 0
counter_inf_k = 0
for i in range(len(probs_bi_korean)):
    uni = perplexity(probs_uni_korean[i])
    bi = perplexity(probs_bi_korean[i])
    perplex_k.append((uni, bi))
    if uni >= bi:
        counter_k += 1
    if uni == float("inf") or bi == float("inf"):
        counter_inf_k += 1

# Telugu
perplex_t = []
counter_t = 0
counter_inf_t = 0
for i in range(len(probs_bi_telugu)):
    uni = perplexity(probs_uni_telugu[i])
    bi = perplexity(probs_bi_telugu[i])
    perplex_t.append((uni, bi))
    if uni >= bi:
        counter_t += 1
    if uni == float("inf") or bi == float("inf"):
        counter_inf_t += 1

# Arabic
perplex_a = []
counter_a = 0
counter_inf = 0
for i in range(len(probs_bi_arabic)):
    uni = perplexity(probs_uni_arabic[i])
    bi = perplexity(probs_bi_arabic[i])
    perplex_a.append((uni, bi))
    if uni >= bi:
        counter_a += 1
    if uni == float("inf") or bi == float("inf"):
        counter_inf += 1

# Context
perplex_c = []
counter_c = 0
counter_inf_c = 0
for i in range(len(probs_bi_context)):  
    uni = perplexity(probs_uni_context[i]) 
    bi = perplexity(probs_bi_context[i])
    perplex_c.append((uni, bi))
    if uni >= bi:
        counter_c += 1
    if uni == float("inf") or bi == float("inf"):
        counter_inf_c += 1


/var/folders/wp/3lqqvgpd7mz76kqxnc2932700000gn/T/ipykernel_82284/3516201195.py:5: RuntimeWarning: divide by zero encountered in scalar divide
  perplexity = (1/probs.prod())**(1/N)
/var/folders/wp/3lqqvgpd7mz76kqxnc2932700000gn/T/ipykernel_82284/3516201195.py:5: RuntimeWarning: overflow encountered in scalar divide
  perplexity = (1/probs.prod())**(1/N)


In [70]:
for index, instance in enumerate(perplex_t[5:10]):
    print(instance, len(probs_uni_telugu[index+5]))

(1863.7951806727701, 1589.6071203338863) 6
(2128.2584447831105, 1872.471376761082) 6
(3385.4574568153066, 1127.4074820680733) 5
(696.4510376499809, 446.7889708243594) 6
(3576.864058482206, 1067.755440483348) 8
